In [5]:
import pandas as pd
import numpy as np

# ============================================================
# PATHS
# ============================================================

fasta_path = "/home/nacho/HDD16/Nacho/RepliCOOC/derep/louvain_by_level_from_table_including_isolates/all_representatives.fasta"
replicon_path = "/home/nacho/HDD16/Nacho/RepliCOOC/derep/plasmid_replicons.tsv"
matrix_hits_path = "/home/nacho/HDD16/Nacho/RepliCOOC/derep/blast_things_all_plasmids/plasmid_query_hit_matrix.tsv"

# ============================================================
# 1) READ ALL PLASMID IDS FROM FASTA
# ============================================================

all_ids = []
with open(fasta_path) as f:
    for line in f:
        if line.startswith(">"):
            pid = line[1:].strip().split()[0]
            all_ids.append(pid)

all_plasmids = pd.DataFrame({"plasmid": all_ids})
print("Total plasmids in FASTA:", all_plasmids.shape[0])

# ============================================================
# 2) LOAD HIT MATRIX (ONLY POSITIVES) AND EXPAND TO FULL UNIVERSE
# ============================================================

mat = pd.read_csv(matrix_hits_path, sep="\t")
for c in ["contig_1", "contig_2", "contig_3"]:
    mat[c] = pd.to_numeric(mat[c], errors="coerce").fillna(0).astype(int)

full_mat = all_plasmids.merge(mat, on="plasmid", how="left")
for c in ["contig_1", "contig_2", "contig_3"]:
    full_mat[c] = full_mat[c].fillna(0).astype(int)

print("Expanded matrix rows:", full_mat.shape[0])
print("Plasmids with any hit:", (full_mat[["contig_1", "contig_2", "contig_3"]].sum(axis=1) > 0).sum())
print("Plasmids with no hits:", (full_mat[["contig_1", "contig_2", "contig_3"]].sum(axis=1) == 0).sum())

# ============================================================
# 3) LOAD REPLICON TABLE AND MERGE
# ============================================================

rep = pd.read_csv(replicon_path, sep="\t")
rep["plasmid_id"] = rep["plasmid_id"].astype(str)
rep["n_replicons"] = pd.to_numeric(rep["n_replicons"], errors="coerce")

df = full_mat.merge(
    rep[["plasmid_id", "replicon_types", "n_replicons", "size_bp"]],
    left_on="plasmid",
    right_on="plasmid_id",
    how="left"
)

print("Merged rows:", df.shape[0])
print("Without replicon annotation:", df["n_replicons"].isna().sum())

# si quieres restringir solo a plasmidos con replicon anotado:
df = df.dropna(subset=["n_replicons"]).copy()
df["n_replicons"] = df["n_replicons"].astype(int)
df["replicon_group"] = np.where(df["n_replicons"] == 1, "single", "multi")

# ============================================================
# 4) DERIVED VARIABLES
# ============================================================

for c in ["contig_1", "contig_2", "contig_3"]:
    df[f"{c}_present"] = (df[c] > 0).astype(int)

df["any_contig_present"] = df[["contig_1_present", "contig_2_present", "contig_3_present"]].max(axis=1)
df["n_contigs_present"] = df[["contig_1_present", "contig_2_present", "contig_3_present"]].sum(axis=1)
df["total_hits"] = df[["contig_1", "contig_2", "contig_3"]].sum(axis=1)

df.to_csv("full_universe_plasmids_contig_hits_vs_replicons.tsv", sep="\t", index=False)

print(df["replicon_group"].value_counts())
print(df[["any_contig_present", "n_contigs_present", "total_hits"]].describe())

Total plasmids in FASTA: 23925
Expanded matrix rows: 23925
Plasmids with any hit: 5982
Plasmids with no hits: 17943
Merged rows: 23925
Without replicon annotation: 0
replicon_group
single    16376
multi      7549
Name: count, dtype: int64
       any_contig_present  n_contigs_present    total_hits
count        23925.000000       23925.000000  23925.000000
mean             0.250031           0.259352      1.092079
std              0.433040           0.459063      2.480017
min              0.000000           0.000000      0.000000
25%              0.000000           0.000000      0.000000
50%              0.000000           0.000000      0.000000
75%              1.000000           1.000000      1.000000
max              1.000000           2.000000     39.000000


In [6]:
# ============================================================
# PLOTS FOR FULL UNIVERSE
# ============================================================

import os
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", context="talk")
outdir = "plots_full_universe"
os.makedirs(outdir, exist_ok=True)

# ------------------------------------------------------------
# 1) prevalence bars
# ------------------------------------------------------------

prev_rows = []
for grp, sub in df.groupby("replicon_group"):
    n = len(sub)
    for contig in ["contig_1_present", "contig_2_present", "contig_3_present", "any_contig_present"]:
        k = sub[contig].sum()
        prev_rows.append({
            "group": grp,
            "feature": contig.replace("_present", ""),
            "n": n,
            "k": k,
            "percent": 100 * k / n
        })

prev = pd.DataFrame(prev_rows)

plt.figure(figsize=(8, 6))
ax = sns.barplot(data=prev, x="feature", y="percent", hue="group")
ax.set_xlabel("")
ax.set_ylabel("Plasmids positive (%)")
for p, (_, row) in zip(ax.patches, prev.iterrows()):
    ax.text(
        p.get_x() + p.get_width()/2,
        p.get_height() + 0.15,
        f"{row['k']}/{row['n']}",
        ha="center", va="bottom", fontsize=10, rotation=90
    )
plt.tight_layout()
plt.savefig(f"{outdir}/prevalence_single_vs_multi.png", dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# 2) total hits among positives only
# ------------------------------------------------------------

pos = df[df["any_contig_present"] == 1].copy()

plt.figure(figsize=(7, 6))
ax = sns.boxplot(data=pos, x="replicon_group", y="total_hits", showfliers=False)
sns.stripplot(data=pos, x="replicon_group", y="total_hits", alpha=0.35, ax=ax)
ax.set_xlabel("")
ax.set_ylabel("Total hits per positive plasmid")
plt.tight_layout()
plt.savefig(f"{outdir}/total_hits_positive_only_single_vs_multi.png", dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# 3) number of replicons in positives vs negatives
# ------------------------------------------------------------

df["hit_status"] = np.where(df["any_contig_present"] == 1, "positive", "negative")

plt.figure(figsize=(8, 6))
ax = sns.boxplot(data=df, x="hit_status", y="n_replicons", showfliers=False)
sns.stripplot(data=df, x="hit_status", y="n_replicons", alpha=0.25, ax=ax)
ax.set_xlabel("")
ax.set_ylabel("Number of replicons")
plt.tight_layout()
plt.savefig(f"{outdir}/n_replicons_positive_vs_negative.png", dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# 4) stacked composition of hit patterns among positives
# ------------------------------------------------------------

pos["pattern"] = (
    pos["contig_1_present"].astype(str) +
    pos["contig_2_present"].astype(str) +
    pos["contig_3_present"].astype(str)
)

tab = pd.crosstab(pos["replicon_group"], pos["pattern"], normalize="index") * 100
tab = tab.reindex(index=["single", "multi"], fill_value=0)

tab.plot(kind="bar", stacked=True, figsize=(9, 6))
plt.ylabel("Positive plasmids (%)")
plt.xlabel("")
plt.legend(title="Pattern", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{outdir}/pattern_composition_positive_only.png", dpi=300, bbox_inches="tight")
plt.close()

print("Saved plots in", outdir)

Saved plots in plots_full_universe
